# <font color="#418FDE" size="6.5" uppercase>**Robust Engineering Scripts**</font>

>Last update: 20260608.
    
By the end of this Lecture, you will be able to:
- Validate user inputs to prevent invalid electrical quantities from corrupting calculations. 
- Use exception handling to manage conversion errors, missing files, and impossible operations. 
- Create simple test cases that confirm engineering functions return expected results. 


## **1. Input Validation**

### **1.1. Parameter Checks**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Electrical Engineers/Module_05/Lecture_A/image_01_01.jpg?v=1780965682" width="250">



>* Confirm inputs match expected electrical quantities
>* Check presence, number format, and meaning

>* Check units, types, and missing values
>* Reject unclear data before calculations

>* Catch input mistakes before calculations spread
>* Stop unclear values to reduce engineering risk



In [ ]:
#@title Python Code - Parameter Checks

# Parameter checks protect engineering calculations.
# Electrical values need numeric meaning.
# Bad inputs should fail early.

# Define accepted units for each parameter.
voltage_units = {"V": 1.0, "kV": 1000.0}
current_units = {"A": 1.0, "mA": 0.001}
length_units = {"m": 1.0, "ft": 0.3048}

# Convert one engineering parameter safely.
def checked_quantity(name, raw_value, raw_unit, allowed_units, minimum):
    try:
        numeric_value = float(raw_value)
    except (TypeError, ValueError):
        raise ValueError(f"{name} must be numeric.")

    if raw_unit not in allowed_units:
        raise ValueError(f"{name} unit is invalid.")
    if numeric_value < minimum:
        raise ValueError(f"{name} is below allowed range.")

    converted_value = numeric_value * allowed_units[raw_unit]
    return converted_value

# Compute a simple single-phase load current.
def load_current(voltage_value, voltage_unit, power_watts):
    voltage = checked_quantity(
        "Voltage", voltage_value, voltage_unit, voltage_units, 1
    )

    power = checked_quantity(
        "Power", power_watts, "W", {"W": 1.0}, 0
    )
    current = power / voltage
    return current

# Prepare valid and invalid test cases.
test_cases = [
    ("valid metric", 230, "V", 1150),
    ("valid kilovolt", 0.23, "kV", 1150),
    ("text voltage", "two thirty", "V", 1150),
    ("negative voltage", -230, "V", 1150),
    ("wrong unit", 230, "amps", 1150),
]

# Run each case without stopping the script.
results = []
for label, volts, unit, watts in test_cases:
    try:
        amps = load_current(volts, unit, watts)
        message = f"{label}: {amps:.2f} A accepted"
    except ValueError as error:
        message = f"{label}: rejected, {error}"

    results.append(message)

# Print compact results for the lecture.
for message in results:
    print(message)

# Check cable length using metric and imperial units.
metric_length = checked_quantity(
    "Cable length", 30, "m", length_units, 0
)

imperial_length = checked_quantity(
    "Cable length", 100, "ft", length_units, 0
)

# Show unit conversion consistency briefly.
print(f"30 m equals {metric_length:.1f} m checked")
print(f"100 ft equals {imperial_length:.1f} m checked")



### **1.2. Acceptable Value Ranges**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Electrical Engineers/Module_05/Lecture_A/image_01_02.jpg?v=1780965681" width="250">



>* Check values match real circuit possibilities
>* Reject impossible inputs before calculations

>* Ranges come from real equipment limits
>* Scripts apply engineering knowledge to reject unsafe inputs

>* Clear range errors guide quick user correction
>* Limits prevent bad calculations and reveal faults



In [ ]:
#@title Python Code - Acceptable Value Ranges

# Validate values before electrical calculations.
# Range limits encode engineering expectations.
# Rejected inputs protect downstream automation.

# Define acceptable engineering ranges.
voltage_range = (0.0, 24.0)
resistance_range = (1.0, 10000.0)
current_range = (0.0, 2.0)

# Store example readings from instruments.
readings = [
    ("Sensor A", 5.0, 1000.0),
    ("Sensor B", -3.0, 470.0),
    ("Sensor C", 12.0, 0.0),
    ("Sensor D", 48.0, 100.0),
]

# Check whether one value fits limits.
def in_range(value, limits):
    low, high = limits
    return low <= value <= high

# Calculate current only after validation.
def safe_current(voltage, resistance):
    if not in_range(voltage, voltage_range):
        raise ValueError("Voltage outside expected low-voltage range")
    if not in_range(resistance, resistance_range):
        raise ValueError("Resistance outside passive resistor range")
    current = voltage / resistance
    if not in_range(current, current_range):
        raise ValueError("Calculated current exceeds project limit")
    return current

# Process each reading safely.
accepted = []
rejected = []
for name, voltage, resistance in readings:
    try:
        current = safe_current(voltage, resistance)
        accepted.append((name, current))
    except ValueError as error:
        rejected.append((name, str(error)))

# Report compact engineering results.
print("Accepted readings:")
for name, current in accepted:
    print(f"{name}: {current:.4f} A")

# Report rejected readings clearly.
print("Rejected readings:")
for name, reason in rejected:
    print(f"{name}: {reason}")

# Confirm expected behavior with simple tests.
assert abs(safe_current(10.0, 1000.0) - 0.01) < 1e-12
assert len(accepted) == 1 and len(rejected) == 3
print("Range validation tests passed.")



### **1.3. Range Validation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Electrical Engineers/Module_05/Lecture_A/image_01_03.jpg?v=1780965684" width="250">



>* Check values stay within safe engineering limits
>* Reject realistic-looking inputs that corrupt results

>* Set realistic limits for each quantity
>* Match ranges to system context

>* Catch range errors near their source
>* Separate real faults from entry mistakes



In [ ]:
#@title Python Code - Range Validation

# Range checks protect engineering calculations.
# Invalid values should fail early.
# Clear messages improve script reliability.

# Define acceptable voltage limits.
MIN_VOLTAGE_V = 0.0
MAX_VOLTAGE_V = 600.0

# Define acceptable resistance limits.
MIN_RESISTANCE_OHM = 0.1
MAX_RESISTANCE_OHM = 10000.0

# Validate one numeric engineering quantity.
def validate_range(name, value, low, high):
    # Reject values outside engineering limits.
    if low <= value <= high:
        return value

    # Explain the failed validation clearly.
    message = f"{name}={value} outside {low} to {high}"
    raise ValueError(message)

# Calculate power after successful validation.
def calculate_power(voltage_v, resistance_ohm):
    # Validate voltage before using equation.
    voltage_v = validate_range(
        "voltage_v", voltage_v, MIN_VOLTAGE_V, MAX_VOLTAGE_V
    )

    # Validate resistance before division.
    resistance_ohm = validate_range(
        "resistance_ohm", resistance_ohm, MIN_RESISTANCE_OHM, MAX_RESISTANCE_OHM
    )

    # Compute power using Ohm law.
    return voltage_v ** 2 / resistance_ohm

# Create small test cases.
test_cases = [
    (120.0, 240.0),
    (480.0, 12.0),
    (700.0, 50.0),
    (24.0, 0.02),
]

# Evaluate each case defensively.
for voltage_v, resistance_ohm in test_cases:
    try:
        power_w = calculate_power(voltage_v, resistance_ohm)
        print(f"OK: {voltage_v} V, {resistance_ohm} ohm -> {power_w:.1f} W")
    except ValueError as error:
        print(f"Rejected: {error}")

# Confirm one expected engineering result.
expected_w = 60.0
actual_w = calculate_power(120.0, 240.0)

# Report whether the simple test passed.
print("Test passed:", abs(actual_w - expected_w) < 1e-9)



## **2. Error Handling**

### **2.1. Exception Handling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Electrical Engineers/Module_05/Lecture_A/image_02_01.jpg?v=1780965669" width="250">



>* Plan for errors in engineering scripts
>* Report failures before invalid calculations continue

>* Handle predictable input mistakes clearly
>* Expose unexpected faults for repair

>* Catch bad data without hiding problems
>* Protect reliable engineering records and decisions



In [ ]:
#@title Python Code - Exception Handling

# Demonstrate exception handling for engineering input data.
# Bad values should not corrupt later calculations.
# Specific exceptions make script failures understandable.

# Define sample entries from field measurement forms.
raw_resistances = ["12.5", "bad", "0", "8.2"]

# Convert text carefully before using engineering formulas.
def safe_power(voltage_text, resistance_text):
    try:
        voltage_value = float(voltage_text)
        resistance_value = float(resistance_text)

        if resistance_value <= 0:
            raise ZeroDivisionError("resistance must be positive")

        power_value = voltage_value ** 2 / resistance_value
        return round(power_value, 2)

    except ValueError:
        return "conversion error"

    except ZeroDivisionError as problem:
        return str(problem)

# Demonstrate several realistic automation outcomes.
print("Exception handling results:")

# Use one voltage and several resistance entries.
for resistance_entry in raw_resistances:
    result_value = safe_power("24", resistance_entry)

    print(f"R={resistance_entry} ohm -> {result_value}")

# Handle a missing configuration file gracefully.
try:
    with open("missing_panel_config.txt", "r") as file_object:
        config_text = file_object.read()

except FileNotFoundError:
    config_text = "default voltage limit used"

print(f"File check -> {config_text}")

# Confirm one known engineering result with a simple test.
expected_value = 57.6
actual_value = safe_power("24", "10")

print(f"Test passed -> {actual_value == expected_value}")



### **2.2. Missing File Handling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Electrical Engineers/Module_05/Lecture_A/image_02_02.jpg?v=1780965670" width="250">



>* External files need reliable error handling
>* Report missing files before unsafe calculations

>* Expect file paths and names to change
>* Report missing files with helpful guidance

>* Choose safe defaults or stop clearly
>* Log missing files for later diagnosis



In [ ]:
#@title Python Code - Missing File Handling

# Missing files should be handled clearly.
# Engineering scripts often depend on data.
# This example uses a safe fallback.

# Import pathlib for reliable file paths.
from pathlib import Path

# Define expected and backup file paths.
expected_path = Path("feeder_load_profile.csv")
backup_path = Path("default_load_profile.csv")

# Create a tiny backup file.
backup_text = "hour,current_a\n0,12.5\n1,13.1\n"
backup_path.write_text(backup_text, encoding="utf-8")

# Read the preferred file safely.
def read_load_profile(file_path, fallback_path):
    # Try the requested engineering data file.
    try:
        data_text = file_path.read_text(encoding="utf-8")
        source_name = file_path.name

    # Handle a missing requested data file.
    except FileNotFoundError:
        message = f"Missing file: {file_path.name}"
        data_text = fallback_path.read_text(encoding="utf-8")
        source_name = fallback_path.name

    # Return both data and source details.
    else:
        message = "Requested file loaded successfully."
    return data_text, source_name, message

# Parse simple comma separated current data.
def average_current_from_csv(csv_text):
    # Split text into nonempty data lines.
    rows = csv_text.strip().splitlines()
    if len(rows) < 2:
        raise ValueError("No measurement rows found.")

    # Convert current column values safely.
    currents = []
    for row in rows[1:]:
        hour_text, current_text = row.split(",")
        currents.append(float(current_text))

    # Prevent division by zero operation.
    if len(currents) == 0:
        raise ZeroDivisionError("No currents available.")
    return sum(currents) / len(currents)

# Run the missing file demonstration.
profile_text, source, note = read_load_profile(expected_path, backup_path)
average_current = average_current_from_csv(profile_text)

# Print concise engineering results.
print("Missing File Handling Demo")
print(note)
print(f"Data source used: {source}")
print(f"Average feeder current: {average_current:.2f} A")
print("Script continued using a documented fallback file.")



### **2.3. Handling Impossible Operations**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Electrical Engineers/Module_05/Lecture_A/image_02_03.jpg?v=1780965672" width="250">



>* Catch calculations that cannot make sense
>* Stop unsafe results and explain corrections

>* Check mathematical and engineering limits.
>* Explain errors before accepting impossible results.

>* Fail safely on impossible engineering conditions
>* Report clear errors for easier debugging



In [ ]:
#@title Python Code - Handling Impossible Operations

# This script handles impossible electrical calculations.
# It uses exceptions for unsafe operations.
# Small tests confirm expected engineering behavior.

# No extra packages are required here.

# Define a guarded resistance calculation.
def calculate_resistance(voltage, current):
    if current == 0:
        raise ValueError("Current cannot be zero for resistance.")

    if voltage < 0:
        raise ValueError("Voltage must be nonnegative here.")

    return voltage / current

# Define a guarded power calculation.
def calculate_power(current, resistance):
    if resistance < 0:
        raise ValueError("Resistance cannot be negative physically.")

    return current ** 2 * resistance

# Store example engineering scenarios.
scenarios = [
    ("valid resistance", calculate_resistance, (12, 2)),
    ("zero current", calculate_resistance, (12, 0)),
    ("negative resistance", calculate_power, (3, -4)),
]

# Run each scenario safely.
for name, function, values in scenarios:
    try:
        result = function(*values)

        print(f"{name}: result = {result:.2f}")
    except ValueError as error:
        print(f"{name}: blocked -> {error}")

# Create tiny checks for expected behavior.
tests_passed = 0
expected = calculate_resistance(24, 6)

# Check a normal engineering result.
if expected == 4:
    tests_passed += 1

# Check that impossible division is rejected.
try:
    calculate_resistance(5, 0)
except ValueError:
    tests_passed += 1

# Check that impossible resistance is rejected.
try:
    calculate_power(2, -1)
except ValueError:
    tests_passed += 1

print(f"Tests passed: {tests_passed} of 3")



## **3. Basic Function Testing**

### **3.1. Expected Results**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Electrical Engineers/Module_05/Lecture_A/image_03_01.jpg?v=1780965675" width="250">



>* Use known outputs as testing standards
>* Start with simple, verifiable input cases

>* Choose realistic tests with predictable answers
>* Catch unit, assumption, and rounding mistakes

>* Tests catch breakages after script changes
>* Expected results build reliable engineering automation



In [ ]:
#@title Python Code - Expected Results

# Basic tests compare functions with known results.
# Expected results catch engineering calculation mistakes.
# Small examples build confidence before automation.

# Define an Ohm law current function.
def current_from_voltage_resistance(volts, ohms):
    if ohms <= 0:
        raise ValueError("Resistance must be positive.")

    return volts / ohms

# Define a power calculation function.
def power_from_voltage_current(volts, amps):
    if volts < 0 or amps < 0:
        raise ValueError("Electrical quantities must be nonnegative.")

    return volts * amps

# Store inputs with expected results.
test_cases = [
    ("current", current_from_voltage_resistance, (120, 60), 2.0),
    ("current", current_from_voltage_resistance, (24, 12), 2.0),
]

# Add power tests with known answers.
test_cases.extend([
    ("power", power_from_voltage_current, (12, 3), 36.0),
    ("power", power_from_voltage_current, (230, 0.5), 115.0),
])

# Compare calculated values against expected values.
passed_tests = 0
total_tests = len(test_cases)
tolerance = 1e-9

# Run each test without noisy output.
for name, function, inputs, expected in test_cases:
    calculated = function(*inputs)
    difference = abs(calculated - expected)

    if difference <= tolerance:
        passed_tests += 1

# Demonstrate one expected exception test.
try:
    current_from_voltage_resistance(120, 0)
except ValueError:
    passed_tests += 1

# Count the exception test separately.
total_tests += 1
print(f"Passed tests: {passed_tests} of {total_tests}")
print("Expected current for 120 V and 60 ohms is 2 A.")
print("Expected power for 12 V and 3 A is 36 W.")
print("Zero resistance correctly raised a ValueError.")



### **3.2. Expected Results**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Electrical Engineers/Module_05/Lecture_A/image_03_02.jpg?v=1780965676" width="250">



>* Know correct answers before testing functions
>* Mismatches reveal calculation or input problems

>* Use realistic expected results to catch quiet errors
>* Treat script calculations as claims needing checks

>* Tests clarify intended function behavior
>* Repeatable checks protect against accidental changes



In [ ]:
#@title Python Code - Expected Results

# Expected results make engineering tests concrete.
# Simple known cases reveal calculation mistakes.
# Passing tests increase script confidence.

# Define a power calculation function.
def calculate_power_watts(voltage_volts, current_amps):

    # Reject impossible negative electrical inputs.
    if voltage_volts < 0 or current_amps < 0:

        raise ValueError("Voltage and current must be nonnegative.")

    # Return power using P equals VI.
    return voltage_volts * current_amps

# Define a unit conversion function.
def milliamps_to_amps(current_milliamps):

    # Convert milliamps to amps safely.
    return current_milliamps / 1000

# Store tests with expected answers.
test_cases = [

    ("power small load", calculate_power_watts, (12, 2), 24),
    ("power zero current", calculate_power_watts, (120, 0), 0),

    ("current conversion", milliamps_to_amps, (250,), 0.25),
]

# Track how many tests pass.
passed_tests = 0

# Run each test case quietly.
for name, function, inputs, expected in test_cases:

    # Compare actual result with expected result.
    actual = function(*inputs)

    # Use tolerance for decimal comparisons.
    passed = abs(actual - expected) < 1e-9

    # Count successful test results.
    passed_tests += int(passed)

    # Print one compact test summary.
    print(f"{name}: expected {expected}, got {actual}, pass={passed}")

# Demonstrate testing an expected exception.
try:

    calculate_power_watts(-5, 2)
    negative_test_passed = False

except ValueError:

    negative_test_passed = True

# Print final compact testing summary.
print(f"Expected error test passed: {negative_test_passed}")

# Show the overall number passed.
print(f"Normal tests passed: {passed_tests} of {len(test_cases)}")



### **3.3. Expected Results**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Electrical Engineers/Module_05/Lecture_A/image_03_03.jpg?v=1780965678" width="250">



>* Know correct answers before testing functions
>* Check representative cases against trusted results

>* Use independently verified expected results
>* Test realistic cases and important boundaries

>* Tests reveal changes after script updates
>* Expected results protect engineering decisions



In [ ]:
#@title Python Code - Expected Results

# Expected results make engineering tests trustworthy.
# Simple cases should be independently verifiable.
# Passing tests protect future script changes.

# Define a small engineering conversion function.
def kw_to_watts(kw_value):

    # Reject impossible negative power values.
    if kw_value < 0:
        raise ValueError("Power cannot be negative.")

    # Return the expected unit conversion.
    return kw_value * 1000

# Define a simple voltage drop calculation.
def voltage_drop(current_amp, resistance_ohm):

    # Validate both electrical input quantities.
    if current_amp < 0 or resistance_ohm < 0:
        raise ValueError("Current and resistance must be nonnegative.")

    # Use Ohm's law for voltage drop.
    return current_amp * resistance_ohm

# Store tests with trusted expected results.
tests = [
    ("2.5 kW converts to watts", kw_to_watts, (2.5,), 2500),
    ("Zero kW converts to zero", kw_to_watts, (0,), 0),
    ("12 A across 0.5 ohm", voltage_drop, (12, 0.5), 6),
    ("Zero current gives zero drop", voltage_drop, (0, 3), 0),
]

# Count how many tests behave correctly.
passed_count = 0

# Compare each actual result against expectation.
for name, function, arguments, expected in tests:

    # Call the function using prepared arguments.
    actual = function(*arguments)

    # Check whether actual and expected match.
    test_passed = actual == expected

    # Update the passing test counter.
    passed_count += int(test_passed)

    # Print one compact test result line.
    print(f"{name}: expected {expected}, got {actual}, pass={test_passed}")

# Show the overall testing summary.
summary = f"Passed {passed_count} of {len(tests)} expected-result tests."
print(summary)



# <font color="#418FDE" size="6.5" uppercase>**Robust Engineering Scripts**</font>


In this lecture, you learned to:
- Validate user inputs to prevent invalid electrical quantities from corrupting calculations. 
- Use exception handling to manage conversion errors, missing files, and impossible operations. 
- Create simple test cases that confirm engineering functions return expected results. 

<font color='yellow'>Congratulations on completing this course!</font>